# Bank Transactions - Clustering & Classification
Notebook ini mengikuti kriteria tugas:
1. Load Dataset & EDA
2. Data Cleaning & Preprocessing
3. Clustering (K-Means)
4. Interpretasi Hasil Clustering
5. Klasifikasi (Decision Tree)

Jalankan cell secara berurutan dari atas ke bawah. Pastikan file `bank_transactions_data_edited.csv` berada di folder yang sama (di Colab, upload ke `/content/`).

In [ ]:
# Install dependency tambahan yang belum ada di Colab secara default.
# scikit-learn dipin ke versi <1.5 karena yellowbrick 1.5 belum kompatibel
# dengan versi scikit-learn terbaru (akan error saat membuat KElbowVisualizer).
!pip install yellowbrick -q
!pip install "scikit-learn>=1.3,<1.5" -q

# PENTING: setelah cell ini selesai dijalankan pertama kali, klik
# Runtime > Restart session (khusus di Google Colab), lalu lanjutkan
# menjalankan cell-cell berikutnya mulai dari cell import di bawah.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report,
    confusion_matrix
)
import joblib

pd.set_option('display.max_columns', None)

## Kriteria 1: Memuat Dataset dan Melakukan Exploratory Data Analysis (EDA)

In [ ]:
# Jika menjalankan di Google Colab, upload file csv terlebih dahulu, atau mount Google Drive
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('bank_transactions_data_edited.csv')
df.head()

In [ ]:
# Informasi dataset
df.info()

In [ ]:
# Statistik deskriptif dataset
df.describe(include='all')

## Kriteria 2: Pembersihan dan Pra Pemrosesan Data

In [ ]:
# Mengecek missing value dan duplikasi data
print("Jumlah missing value per kolom:")
print(df.isnull().sum())
print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
# Menangani data yang hilang
df = df.dropna()

# Menghapus data duplikat
df = df.drop_duplicates()

print("Shape setelah dropna & drop_duplicates:", df.shape)

In [ ]:
# Drop kolom yang bersifat ID, Address, dan Date karena tidak relevan untuk clustering/klasifikasi
# (TransactionID, AccountID, DeviceID -> ID; IP Address, Location -> Address;
#  TransactionDate, PreviousTransactionDate -> Date)
cols_to_drop = [
    'TransactionID', 'AccountID', 'DeviceID', 'IP Address',
    'MerchantID', 'TransactionDate', 'PreviousTransactionDate', 'Location'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df.head()

In [ ]:
# Feature encoding untuk fitur kategorikal menggunakan LabelEncoder
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Kolom kategorikal:", categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df.head()

## Kriteria 3: Membangun Model Clustering

In [ ]:
# Standarisasi fitur numerik agar skala seragam sebelum clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
X_scaled = pd.DataFrame(X_scaled, columns=df.columns, index=df.index)
X_scaled.head()

In [ ]:
# Menentukan jumlah cluster terbaik menggunakan Elbow Method
model_kmeans = KMeans(random_state=42, n_init=10)
visualizer = KElbowVisualizer(model_kmeans, k=(2, 11), timing=False)
visualizer.fit(X_scaled)
visualizer.show()

best_k = visualizer.elbow_value_
print("Jumlah cluster optimal (elbow):", best_k)

In [ ]:
# Fallback jika elbow tidak terdeteksi otomatis
if best_k is None:
    best_k = 3
    print("Elbow tidak terdeteksi otomatis, menggunakan default k =", best_k)

In [ ]:
# Membangun model clustering dengan K-Means menggunakan jumlah cluster terbaik
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df['Cluster'] = cluster_labels
df.head()

In [ ]:
# Menyimpan model clustering agar dapat dinilai otomatis oleh reviewer
joblib.dump(kmeans, 'model_clustering.pkl')
print("Model clustering berhasil disimpan sebagai model_clustering.pkl")

## Kriteria 4: Interpretasi Hasil Clustering

In [ ]:
# Analisis deskriptif (mean, min, max) untuk tiap cluster pada fitur numerik
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'Cluster']

cluster_summary = df.groupby('Cluster')[numeric_cols].agg(['mean', 'min', 'max'])
cluster_summary

In [ ]:
# Jumlah anggota tiap cluster
df['Cluster'].value_counts().sort_index()

**Karakteristik tiap cluster** (isi/analisis sesuai output tabel di atas):

- Cluster 0: ...
- Cluster 1: ...
- Cluster 2: ...

Silakan lengkapi narasi karakteristik cluster berdasarkan nilai mean/min/max yang muncul pada tabel `cluster_summary` di atas (misalnya cluster dengan rata-rata TransactionAmount tinggi & AccountBalance rendah, dsb).

In [ ]:
# Mengekspor data hasil preprocessing + hasil clustering
# Kolom hasil cluster diberi nama 'Target' agar bisa digunakan sebagai label pada tahap klasifikasi
df_train = df.copy()
df_train = df_train.rename(columns={'Cluster': 'Target'})
df_train.to_csv('data_train_clustered.csv', index=False)

df_train.head()

## Kriteria 5: Membangun Model Klasifikasi

In [ ]:
# Memisahkan fitur (X) dan target (y) dari hasil clustering
X = df_train.drop(columns=['Target'])
y = df_train['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah data training:", X_train.shape)
print("Jumlah data testing :", X_test.shape)

In [ ]:
# Membangun model klasifikasi dengan algoritma Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)

# Tuning hyperparameter menggunakan GridSearchCV
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(dt_model, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_dt_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

In [ ]:
# Evaluasi model pada data testing
y_pred = best_dt_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Visualisasi confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Decision Tree')
plt.show()

In [ ]:
# Menyimpan model klasifikasi agar dapat dinilai otomatis oleh reviewer
joblib.dump(best_dt_model, 'decision_tree_model.h5')
print("Model klasifikasi berhasil disimpan sebagai decision_tree_model.h5")

## Selesai
File yang dihasilkan:
- `model_clustering.pkl` -> model K-Means
- `data_train_clustered.csv` -> dataset hasil preprocessing + kolom `Target`
- `decision_tree_model.h5` -> model Decision Tree hasil training

Semua file akan tersimpan di direktori kerja Colab (`/content/`). Anda bisa mengunduhnya lewat panel File di sisi kiri Colab, atau menambahkan `from google.colab import files; files.download('nama_file')`.